In [1]:
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

c:\Users\riddh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
docs=[
    Document(page_content="Apples are red in colour. Mangoes are yellow in colour. Harry potter's First movie was THE SORCERER'S STONE.Lychee are pink in colour"),
    Document(page_content="Green vegetables are nutricious and we must eat it. Harry Potter second movie was CHAMBER OF SECRETS."),
    Document(page_content="Harry potter last movie was DEATHLY HALLOWS PARRT 2"),
    Document(page_content="Emma Watson played role of Hermioner Granger in Harry potter movies."),
    Document(page_content="I love MANGOESS. Harry POtter also likes mangoes")
]
embedding = OllamaEmbeddings(model="nomic-embed-text")

vdb=Chroma.from_documents(
    embedding=embedding,
    documents=docs,
    persist_directory="contextualRetriever_db"
)


# Creating llm and retriever for Contextual Compression retriever
#### Compressor will be llm chain extractor and retriever to find documents containing similar content

In [3]:
llm= ChatGroq(model="openai/gpt-oss-20b")

retriever = vdb.as_retriever(
    search_kwargs={"k":5}
)
compressor= LLMChainExtractor.from_llm(llm)

contextualRet=ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor= compressor
) 

result = contextualRet.invoke(
    "Tell me about harry potter "
)

for doc in result:
    print(doc.page_content)

Harry potter last movie was DEATHLY HALLOWS PARRT 2
Emma Watson played role of Hermioner Granger in Harry potter movies.
Harry potter's First movie was THE SORCERER'S STONE.
Extracted relevant parts:
Harry POtter also likes mangoes
Harry Potter second movie was CHAMBER OF SECRETS.
